# 02 · Golden dataset (base dos testes de regressão)

**Objetivo:** criar um conjunto fixo de casos "já validados" (histórico fictício, com **acertos e erros conhecidos**)
contra o qual **toda nova versão** de modelo ou prompt será comparada no LangSmith.

Cada exemplo guarda:

| Campo | Para quê |
|---|---|
| `inputs.pergunta` + `inputs.geladeira` | a entrada do agente (geladeira **congelada** dentro do exemplo) |
| `outputs.tools_esperadas` | base do avaliador de *tool selection accuracy* |
| `outputs.complexidade` (`facil`/`medio`/`dificil`) | permite cruzar **modelo × complexidade** no notebook 05 |
| `outputs.ingredientes_prioritarios` / `ingredientes_vencidos` | fatos objetivos para o juiz (calculados sem LLM) |
| `outputs.resposta_referencia` | critério do que é uma boa resposta |
| `outputs.historico_status` / `historico_observacao` | o caso já foi **acerto** ou **erro** numa versão anterior (e qual erro) |

## Por que congelar?

A geladeira procedural é ótima para **gerar** variedade, mas se ela mudasse a cada execução, uma nota menor
poderia ser culpa do modelo **ou** de uma geladeira mais difícil naquela rodada — impossível separar.

Então: geramos uma vez (seed fixa + data de referência fixa) → salvamos em `data/golden_dataset.json` → subimos para o LangSmith.
Daí em diante, **ninguém edita** este dataset. Mudou o critério? Cria-se `receitas-golden-v2`.

A **data de referência** também fica congelada no exemplo: sem ela, um item que "vence amanhã" hoje estaria vencido semana que vem.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # adiciona a raiz do projeto ao path

from receitas import config

## 1. Os cenários
Três níveis de dificuldade, desenhados para sustentar o roteamento do notebook 05:

In [2]:
import pandas as pd
from receitas.dataset import CENARIOS

pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{
    "id": c["id"], "nível": c["complexidade"], "pergunta": c["pergunta"],
    "tools esperadas": c["tools"], "itens na geladeira": c["n_itens"],
    "histórico": c["historico"][0], "erro conhecido": c["historico"][1],
} for c in CENARIOS])

,id,nível,pergunta,tools esperadas,itens na geladeira,histórico,erro conhecido
0,F1,facil,O que posso cozinhar hoje com o que tenho na geladeira?,"[check_geladeira, criar_receita]","(6, 10)",acerto,
1,F2,facil,"Me sugere um jantar rápido, de uns 20 minutos, com o que tem aqui.","[check_geladeira, criar_receita]","(6, 10)",acerto,
2,F3,facil,Quero um café da manhã reforçado com o que tenho na geladeira.,"[check_geladeira, criar_receita]","(6, 10)",erro,v0 chamou buscar_receitas_similares sem necessidade (custo ~3x maior)
3,F4,facil,Quais itens da minha geladeira estão vencendo?,[verificar_validade],"(8, 12)",erro,v0 chamou check_geladeira + verificar_validade (redundante)
4,F5,facil,Qual a temperatura ideal para manter a geladeira?,[],"(6, 8)",acerto,
5,F6,facil,Dá pra fazer um lanche pra levar pro trabalho com o que tem na geladeira?,"[check_geladeira, criar_receita]","(6, 10)",acerto,
6,M1,medio,"Faz uma receita só com o que está para vencer, não quero desperdiçar nada.","[verificar_validade, criar_receita]","(15, 25)",acerto,
7,M2,medio,Me ajuda a usar o que vence nos próximos dias num almoço?,"[verificar_validade, criar_receita]","(15, 25)",erro,v0 usou um item JÁ vencido na receita
8,M3,medio,"Quero um jantar que aproveite o que está vencendo, mas pode completar com outras coisa...","[check_geladeira, verificar_validade, criar_receita]","(15, 25)",acerto,
9,M4,medio,"Tenho coisas quase estragando aqui, monta uma sopa com elas?","[verificar_validade, criar_receita]","(15, 25)",erro,v0 ignorou a validade e montou a sopa com itens que duram semanas


- **Fácil:** geladeira pequena e sem nada vencendo, pedido direto → `check_geladeira` + `criar_receita`.
  Inclui dois casos de borda: só listar o que vence (`verificar_validade` sozinha) e uma pergunta geral que **não deve usar nenhuma tool** (custo mínimo).
- **Médio:** priorizar o que vai vencer → `verificar_validade` + `criar_receita` (e `check_geladeira` quando o pedido pede para completar com o resto).
- **Difícil:** geladeira grande, pede inspiração em prato conhecido e/ou tem restrições/faltas → RAG + inventário + validade + receita.

Os **erros conhecidos** descrevem falhas típicas: tool desnecessária (custo), item vencido usado, ingrediente inventado.
Ficam no dataset justamente para garantir que não voltem (regressão).

## 2. Gerar (uma vez) e congelar

In [3]:
from receitas.dataset import carregar_ou_criar_golden

exemplos = carregar_ou_criar_golden()  # carrega o JSON congelado (ou cria na primeira execução)
print(f"{len(exemplos)} exemplos | arquivo: {config.ARQ_GOLDEN}")

pd.DataFrame([{
    "id": e["outputs"]["id_cenario"],
    "nível": e["outputs"]["complexidade"],
    "itens": len(e["inputs"]["geladeira"]["itens"]),
    "vencendo (prioritários)": len(e["outputs"]["ingredientes_prioritarios"]),
    "vencidos (proibidos)": len(e["outputs"]["ingredientes_vencidos"]),
    "data de referência": e["inputs"]["geladeira"]["data_referencia"],
    "seed": e["inputs"]["geladeira"]["seed"],
    "histórico": e["outputs"]["historico_status"],
} for e in exemplos])

16 exemplos | arquivo: data/golden_dataset.json


,id,nível,itens,vencendo (prioritários),vencidos (proibidos),data de referência,seed,histórico
0,F1,facil,7,0,0,2026-10-02,2026,acerto
1,F2,facil,8,0,0,2026-10-02,2027,acerto
2,F3,facil,9,0,0,2026-10-02,2028,erro
3,F4,facil,12,3,2,2026-10-02,2029,erro
4,F5,facil,8,0,0,2026-10-02,2030,acerto
5,F6,facil,7,0,0,2026-10-02,2031,acerto
6,M1,medio,23,6,7,2026-10-02,2032,acerto
7,M2,medio,24,7,2,2026-10-02,3033,erro
8,M3,medio,25,11,4,2026-10-02,2034,acerto
9,M4,medio,15,3,1,2026-10-02,3035,erro


In [4]:
import json

exemplo = exemplos[7]  # M2 (erro conhecido)
resumo = {**exemplo, "inputs": {**exemplo["inputs"], "geladeira": {
    **exemplo["inputs"]["geladeira"], "itens": exemplo["inputs"]["geladeira"]["itens"][:3] + ["..."]}}}
print(json.dumps(resumo, ensure_ascii=False, indent=2))

{
  "inputs": {
    "pergunta": "Me ajuda a usar o que vence nos próximos dias num almoço?",
    "geladeira": {
      "seed": 3033,
      "data_referencia": "2026-10-02",
      "itens": [
        {
          "nome": "bife de alcatra",
          "quantidade": 550.0,
          "unidade": "g",
          "categoria": "carnes",
          "validade": "2026-10-07"
        },
        {
          "nome": "peito de frango",
          "quantidade": 700.0,
          "unidade": "g",
          "categoria": "carnes",
          "validade": "2026-10-07"
        },
        {
          "nome": "coentro",
          "quantidade": 1.0,
          "unidade": "maço",
          "categoria": "ervas frescas",
          "validade": "2026-10-08"
        },
        "..."
      ]
    }
  },
  "outputs": {
    "id_cenario": "M2",
    "tools_esperadas": [
      "verificar_validade",
      "criar_receita"
    ],
    "complexidade": "medio",
    "ingredientes_prioritarios": [
      "repolho",
      "pesto",
      "bacalh

### Prova de que está congelado (e reprodutível)

Regerando com a mesma seed e a mesma data de referência, o resultado é idêntico ao arquivo salvo.

In [5]:
from datetime import date
from receitas.dataset import gerar_golden_dataset

data_congelada = date.fromisoformat(exemplos[0]["inputs"]["geladeira"]["data_referencia"])
print("Regenerado == congelado?", gerar_golden_dataset(data_referencia=data_congelada) == exemplos)

Regenerado == congelado? True


## 3. Publicar no LangSmith

Cada exemplo vai com um **split** igual ao nível de complexidade. Isso permite, por exemplo, rodar um experimento só
nos casos difíceis: `evaluate(..., data=client.list_examples(dataset_name=..., splits=["dificil"]))`.

In [6]:
from receitas.dataset import publicar_no_langsmith

print(publicar_no_langsmith(exemplos))

Dataset 'receitas-golden-v1' criado com 16 exemplos.


In [7]:
from collections import Counter
from langsmith import Client

client = Client()
no_langsmith = list(client.list_examples(dataset_name=config.NOME_DATASET))
print("Exemplos no LangSmith:", len(no_langsmith))
print("Por nível:", Counter(e.outputs["complexidade"] for e in no_langsmith))
print("Por histórico:", Counter(e.outputs["historico_status"] for e in no_langsmith))

Exemplos no LangSmith: 16
Por nível: Counter({'facil': 6, 'dificil': 5, 'medio': 5})
Por histórico: Counter({'acerto': 10, 'erro': 6})


## 4. Variância (opcional): rodar cada exemplo N vezes

Mesmo com `temperature=0`, LLMs não são 100% determinísticos. Para saber se um modelo é **consistente** ou **errático**,
o `evaluate()` aceita `num_repetitions=3`: cada exemplo roda 3 vezes no mesmo experimento, e calculamos o desvio-padrão
da nota por exemplo. Isso é usado no notebook 05 (variável `N_REPETICOES`) — custa N× mais, por isso começa em 1.

**Próximo passo:** notebook 03, onde definimos os avaliadores e rodamos o primeiro experimento contra este dataset.